# Resource Utilisation Analysis

Comprehensive analysis of resource consumption across different system activities including Bitbucket ingestion, consistency graph building, document ingestion, and RAG queries.

This notebook examines CPU, memory, GPU, disk, and network metrics to identify patterns, bottlenecks, and optimisation opportunities.

## 1. Import Required Libraries

In [ ]:
import json
import os
import sys
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Tuple
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")
print(f"  - Pandas: {pd.__version__}")
print(f"  - NumPy: {np.__version__}")
print(f"  - Matplotlib: {plt.matplotlib.__version__}")

## 2. Load And Parse Log Files

In [ ]:
# Find all resource_stats JSON files
logs_path = Path.cwd().parent / 'logs' / 'resource_stats'
pattern = str(logs_path / 'resource_stats_*.json')
resource_files = sorted(glob(pattern))

print(f"Found {len(resource_files)} resource stats files")
print(f"Resource stats directory: {logs_path}")
print(f"\nFile count by activity:")

# Count files by activity type
activity_counts: Dict[str, int] = defaultdict(int)
for file in resource_files:
    filename = Path(file).name
    # Extract activity type from filename (e.g., resource_stats_[activity_type]_...)
    parts = filename.replace('resource_stats_', '').split('_')
    # Activity type is everything except the last parts (which are timestamp)
    if 'R_' in filename or filename.count('_') >= 2:
        # For files like 'bitbucket_ingestion_R_20260129...' or 'consistency_graph_build_20260129...'
        if 'bitbucket_ingestion_R' in filename:
            activity = 'bitbucket_ingestion'
        elif 'consistency_graph_build' in filename:
            activity = 'consistency_graph_build'
        elif 'document_ingestion' in filename:
            activity = 'document_ingestion'
        elif 'academic_ingestion' in filename:
            activity = 'academic_ingestion'
        elif 'rag_query' in filename:
            activity = 'rag_query'
        else:
            activity = '_'.join(parts[:-2])
        activity_counts[activity] += 1

for activity, count in sorted(activity_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {activity}: {count} files")

In [ ]:
# Parse all JSON files and create unified dataset
records: List[Dict[str, Any]] = []

for filepath in resource_files:
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
            
        # Extract activity type
        filename = Path(filepath).name
        if 'bitbucket_ingestion_R' in filename:
            activity = 'bitbucket_ingestion'
        elif 'consistency_graph_build' in filename:
            activity = 'consistency_graph_build'
        elif 'document_ingestion' in filename:
            activity = 'document_ingestion'
        elif 'academic_ingestion' in filename:
            activity = 'academic_ingestion'
        elif 'rag_query' in filename:
            activity = 'rag_query'
        else:
            activity = data.get('operation', 'unknown')
        
        # Parse timestamp
        timestamp_str = data.get('timestamp', '')
        try:
            timestamp = pd.to_datetime(timestamp_str)
        except:
            timestamp = None
        
        # Create base record
        record = {
            'activity': activity,
            'timestamp': timestamp,
            'duration_seconds': data.get('duration_seconds', 0),
            'operation': data.get('operation', activity),
        }
        
        # Extract metrics for each process type
        processes = data.get('processes', {})
        for process_name, process_data in processes.items():
            if process_data.get('detected', False):
                prefix = f'{process_name}_'
                record[f'{prefix}cpu_max'] = process_data.get('cpu_percent_max', 0)
                record[f'{prefix}cpu_avg'] = process_data.get('cpu_percent_avg', 0)
                record[f'{prefix}memory_mb_max'] = process_data.get('memory_mb_max', 0)
                record[f'{prefix}memory_mb_avg'] = process_data.get('memory_mb_avg', 0)
                record[f'{prefix}gpu_max'] = process_data.get('gpu_percent_max', 0)
                record[f'{prefix}gpu_avg'] = process_data.get('gpu_percent_avg', 0)
                record[f'{prefix}vram_mb_max'] = process_data.get('vram_mb_max', 0)
                record[f'{prefix}disk_read_mb'] = process_data.get('disk_read_mb_total', 0)
                record[f'{prefix}disk_write_mb'] = process_data.get('disk_write_mb_total', 0)
                record[f'{prefix}network_sent_mb'] = process_data.get('network_sent_mb_total', 0)
                record[f'{prefix}network_recv_mb'] = process_data.get('network_recv_mb_total', 0)
                record[f'{prefix}threads'] = process_data.get('threads_max', 0)
                record[f'{prefix}fds'] = process_data.get('file_descriptors_max', 0)
        
        records.append(record)
    except Exception as e:
        print(f"⚠ Error processing {filepath}: {e}")

# Create DataFrame
df = pd.DataFrame(records)

print(f"\n✓ Loaded {len(df)} records")
valid_timestamps = df['timestamp'].notna().sum()
print(f"Valid timestamps: {valid_timestamps} / {len(df)}")
if valid_timestamps > 0:
    print(f"Date range: {df[df['timestamp'].notna()]['timestamp'].min()} to {df[df['timestamp'].notna()]['timestamp'].max()}")
print(f"Duration range: {df['duration_seconds'].min():.4f}s to {df['duration_seconds'].max():.4f}s")
print(f"\n⚠️  NOTE: Short durations (<1s) result in single-sample measurements where avg = max")

## 3. Explore Data Structure

In [ ]:
# Display basic dataset information
print("📊 Dataset Overview:")
print(f"  Shape: {df.shape}")
print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
print(f"\nActivity types present:")
print(df['activity'].value_counts())

print(f"\n📋 Sample record:")
print(df.iloc[0].to_string())

print(f"\nAvailable metrics (sample columns):")
metric_columns = [col for col in df.columns if any(metric in col for metric in ['cpu', 'memory', 'gpu', 'disk', 'network', 'threads', 'fds'])]
print(f"  Total metric columns: {len(metric_columns)}")
print(f"  Sample metrics: {metric_columns[:10]}")

print(f"\nData types summary:")
print(df.dtypes.value_counts())

print(f"\nMissing values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("  No missing values found")

### 3.1 Data Quality Validation

In [ ]:
# Validate data quality and explain avg=max phenomenon
print("=" * 80)
print("DATA QUALITY VALIDATION")
print("=" * 80)

# Check for avg=max matches (indicates single-sample measurements)
def check_avg_max_match(df: pd.DataFrame, metric_prefix: str) -> int:
    """Count records where avg equals max for a given metric."""
    avg_col = f'{metric_prefix}_avg'
    max_col = f'{metric_prefix}_max'
    if avg_col in df.columns and max_col in df.columns:
        matches = (df[avg_col] == df[max_col]).sum()
        return matches
    return 0

print("\n📊 Measurement Quality Check:")
print("\nRecords where Avg = Max (indicates single-sample measurement):")

cpu_matches = check_avg_max_match(df, 'python_cpu')
mem_matches = check_avg_max_match(df, 'python_memory_mb')
gpu_matches = check_avg_max_match(df, 'python_gpu')

total_records = len(df)
print(f"  CPU:    {cpu_matches:4d} / {total_records} ({cpu_matches/total_records*100:.1f}%)")
print(f"  Memory: {mem_matches:4d} / {total_records} ({mem_matches/total_records*100:.1f}%)")
print(f"  GPU:    {gpu_matches:4d} / {total_records} ({gpu_matches/total_records*100:.1f}%)")

# Analyse durations
print("\n⏱️  Duration Analysis:")
print(f"  Mean duration:   {df['duration_seconds'].mean():.4f}s")
print(f"  Median duration: {df['duration_seconds'].median():.4f}s")
print(f"  Min duration:    {df['duration_seconds'].min():.4f}s")
print(f"  Max duration:    {df['duration_seconds'].max():.4f}s")

short_ops = (df['duration_seconds'] < 1.0).sum()
print(f"\n  Operations < 1 second: {short_ops} / {total_records} ({short_ops/total_records*100:.1f}%)")

if short_ops > total_records * 0.8:
    print("\n⚠️  INSIGHT: Most operations are very short (<1s).")
    print("   With default 1s sampling interval, only 1 sample is captured per operation.")
    print("   This causes avg = max for most metrics.")
    print("\n   [!] RECOMMENDATIONS:")
    print("   1. For longer operations: Monitor over full duration (already happening)")
    print("   2. For short operations: Reduce sampling interval in ResourceMonitor:")
    print("      ResourceMonitor(operation_name='...', interval=0.1)  # 100ms samples")
    print("   3. For snapshot monitoring: Avg=Max is expected and correct")
elif short_ops > 0:
    print("\nℹ️  NOTE: Some short operations (<1s) will show avg=max due to single sampling.")

# Check timestamp validity
print("\n[*] Timestamp Validation:")
valid_ts = df['timestamp'].notna().sum()
invalid_ts = total_records - valid_ts
print(f"  Valid timestamps:   {valid_ts} / {total_records}")
if invalid_ts > 0:
    print(f"  Invalid timestamps: {invalid_ts} / {total_records} (⚠️  will be excluded from time series)")

# Network sent = received check
print("\n[NET] Network I/O Validation:")
if 'python_network_sent_mb' in df.columns and 'python_network_recv_mb' in df.columns:
    net_matches = (df['python_network_sent_mb'] == df['python_network_recv_mb']).sum()
    print(f"  Sent = Received: {net_matches} / {total_records} ({net_matches/total_records*100:.1f}%)")
    if net_matches > total_records * 0.9:
        print("  ℹ️  Most operations show equal sent/received bytes.")
        print("     This may be normal for local operations or balanced request/response patterns.")

print("\n" + "=" * 80)

## 4. Analyse Resource Utilisation By Activity Type

In [ ]:
# Aggregate statistics by activity type
def create_activity_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Create summary statistics for each activity type."""
    activities = df['activity'].unique()
    summaries = []
    
    for activity in activities:
        activity_df = df[df['activity'] == activity]
        
        # Python metrics (most relevant)
        summary = {
            'Activity': activity,
            'Count': len(activity_df),
            'Avg Duration (s)': activity_df['duration_seconds'].mean(),
            'Max Duration (s)': activity_df['duration_seconds'].max(),
            'CPU Max (%)': activity_df['python_cpu_max'].mean(),
            'CPU Avg (%)': activity_df['python_cpu_avg'].mean(),
            'Memory Max (MB)': activity_df['python_memory_mb_max'].mean(),
            'Memory Avg (MB)': activity_df['python_memory_mb_avg'].mean(),
            'GPU Max (%)': activity_df['python_gpu_max'].mean(),
            'GPU Avg (%)': activity_df['python_gpu_avg'].mean(),
            'Disk Read (MB)': activity_df['python_disk_read_mb'].mean(),
            'Disk Write (MB)': activity_df['python_disk_write_mb'].mean(),
            'Network Sent (MB)': activity_df['python_network_sent_mb'].mean(),
            'Network Recv (MB)': activity_df['python_network_recv_mb'].mean(),
            'Threads': activity_df['python_threads'].mean(),
        }
        summaries.append(summary)
    
    return pd.DataFrame(summaries).sort_values('CPU Max (%)', ascending=False)

activity_summary = create_activity_summary(df)

print("📊 Resource Utilisation By Activity Type:")
print(activity_summary.to_string())
print()

# Display detailed statistics
print("\n📈 Detailed Statistics:\n")
for activity in df['activity'].unique():
    activity_df = df[df['activity'] == activity]
    print(f"Activity: {activity}")
    print(f"  Record count: {len(activity_df)}")
    print(f"  Date range: {activity_df['timestamp'].min()} to {activity_df['timestamp'].max()}")
    print(f"  Duration stats (seconds):")
    print(f"    - Mean: {activity_df['duration_seconds'].mean():.6f}")
    print(f"    - Median: {activity_df['duration_seconds'].median():.6f}")
    print(f"    - Min: {activity_df['duration_seconds'].min():.6f}")
    print(f"    - Max: {activity_df['duration_seconds'].max():.6f}")
    print()

## 5. Compare Resource Usage Across Activities

In [ ]:
# Identify resource usage patterns
print("🔍 Resource Usage Patterns:\n")

# Highest CPU activities
cpu_usage = activity_summary.sort_values('CPU Max (%)', ascending=False)
print("Top CPU consumers:")
print(cpu_usage[['Activity', 'CPU Max (%)', 'CPU Avg (%)']].head(3).to_string(index=False))

# Highest memory activities
mem_usage = activity_summary.sort_values('Memory Max (MB)', ascending=False)
print("\n\nTop memory consumers:")
print(mem_usage[['Activity', 'Memory Max (MB)', 'Memory Avg (MB)']].head(3).to_string(index=False))

# Most disk I/O
disk_usage = activity_summary.sort_values('Disk Read (MB)', ascending=False)
print("\n\nTop disk I/O activities:")
print(disk_usage[['Activity', 'Disk Read (MB)', 'Disk Write (MB)']].head(3).to_string(index=False))

# Network usage
net_usage = activity_summary.sort_values('Network Sent (MB)', ascending=False)
print("\n\nTop network users:")
print(net_usage[['Activity', 'Network Sent (MB)', 'Network Recv (MB)']].head(3).to_string(index=False))

# Efficiency analysis
print("\n\n📊 Efficiency Analysis (Resources per second):")
efficiency_data = []
for _, row in activity_summary.iterrows():
    avg_duration = row['Avg Duration (s)']
    if avg_duration > 0:
        efficiency_data.append({
            'Activity': row['Activity'],
            'CPU-seconds/op': row['CPU Avg (%)'] * avg_duration,
            'Memory-MB-seconds/op': row['Memory Avg (MB)'] * avg_duration,
            'Disk-MB/op': row['Disk Read (MB)'] + row['Disk Write (MB)'],
            'Avg Duration (s)': avg_duration,
        })

efficiency_df = pd.DataFrame(efficiency_data).sort_values('CPU-seconds/op', ascending=False)
print(efficiency_df.to_string(index=False))

## 6. Visualise Resource Metrics

In [ ]:
# Create comprehensive visualisations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Resource Utilisation By Activity Type', fontsize=16, fontweight='bold')

# 1. CPU Usage Comparison
ax = axes[0, 0]
activity_summary_sorted = activity_summary.sort_values('CPU Max (%)', ascending=True)
ax.barh(activity_summary_sorted['Activity'], activity_summary_sorted['CPU Max (%)'], color='#FF6B6B')
ax.set_xlabel('Max CPU (%)')
ax.set_title('Peak CPU Usage')
ax.grid(axis='x', alpha=0.3)

# 2. Memory Usage Comparison
ax = axes[0, 1]
activity_summary_sorted = activity_summary.sort_values('Memory Max (MB)', ascending=True)
ax.barh(activity_summary_sorted['Activity'], activity_summary_sorted['Memory Max (MB)'], color='#4ECDC4')
ax.set_xlabel('Max Memory (MB)')
ax.set_title('Peak Memory Usage')
ax.grid(axis='x', alpha=0.3)

# 3. GPU Usage Comparison
ax = axes[0, 2]
activity_summary_sorted = activity_summary.sort_values('GPU Max (%)', ascending=True)
ax.barh(activity_summary_sorted['Activity'], activity_summary_sorted['GPU Max (%)'], color='#FFE66D')
ax.set_xlabel('Max GPU (%)')
ax.set_title('Peak GPU Usage')
ax.grid(axis='x', alpha=0.3)

# 4. Disk I/O Comparison
ax = axes[1, 0]
disk_data = activity_summary.sort_values('Disk Read (MB)', ascending=True)
x = np.arange(len(disk_data))
width = 0.35
ax.barh(x - width/2, disk_data['Disk Read (MB)'], width, label='Read', color='#A8E6CF')
ax.barh(x + width/2, disk_data['Disk Write (MB)'], width, label='Write', color='#FF8B94')
ax.set_xlabel('Data (MB)')
ax.set_title('Average Disk I/O')
ax.set_yticks(x)
ax.set_yticklabels(disk_data['Activity'])
ax.legend()
ax.grid(axis='x', alpha=0.3)

# 5. Network Usage Comparison
ax = axes[1, 1]
net_data = activity_summary.sort_values('Network Sent (MB)', ascending=True)
x = np.arange(len(net_data))
width = 0.35
ax.barh(x - width/2, net_data['Network Sent (MB)'], width, label='Sent', color='#95E1D3')
ax.barh(x + width/2, net_data['Network Recv (MB)'], width, label='Received', color='#F38181')
ax.set_xlabel('Data (MB)')
ax.set_title('Average Network Usage')
ax.set_yticks(x)
ax.set_yticklabels(net_data['Activity'])
ax.legend()
ax.grid(axis='x', alpha=0.3)

# 6. Operation Count
ax = axes[1, 2]
activity_counts = activity_summary.sort_values('Count', ascending=True)
ax.barh(activity_counts['Activity'], activity_counts['Count'], color='#C7CEEA')
ax.set_xlabel('Number Of Operations')
ax.set_title('Activity Frequency')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualisations created")

In [ ]:
# Box plots for resource distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Resource Distribution By Activity Type', fontsize=16, fontweight='bold')

# CPU Distribution
ax = axes[0, 0]
cpu_data = [df[df['activity'] == act]['python_cpu_max'].values for act in df['activity'].unique()]
bp = ax.boxplot(cpu_data, labels=df['activity'].unique(), patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#FF6B6B')
ax.set_ylabel('Max CPU (%)')
ax.set_title('CPU Usage Distribution')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Memory Distribution
ax = axes[0, 1]
mem_data = [df[df['activity'] == act]['python_memory_mb_max'].values for act in df['activity'].unique()]
bp = ax.boxplot(mem_data, labels=df['activity'].unique(), patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#4ECDC4')
ax.set_ylabel('Max Memory (MB)')
ax.set_title('Memory Usage Distribution')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Duration Distribution
ax = axes[1, 0]
dur_data = [df[df['activity'] == act]['duration_seconds'].values for act in df['activity'].unique()]
bp = ax.boxplot(dur_data, labels=df['activity'].unique(), patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#FFE66D')
ax.set_ylabel('Duration (seconds)')
ax.set_title('Operation Duration Distribution')
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# GPU Distribution
ax = axes[1, 1]
gpu_data = [df[df['activity'] == act]['python_gpu_max'].values for act in df['activity'].unique()]
bp = ax.boxplot(gpu_data, labels=df['activity'].unique(), patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#FFB6B9')
ax.set_ylabel('Max GPU (%)')
ax.set_title('GPU Usage Distribution')
ax.grid(axis='y', alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("✓ Distribution visualisations created")

## 7. Generate Summary Statistics

In [ ]:
# Generate comprehensive summary tables
print("="*80)
print("COMPREHENSIVE RESOURCE UTILISATION SUMMARY")
print("="*80)

# Total consumption analysis
print("\n📊 TOTAL RESOURCE CONSUMPTION BY ACTIVITY:\n")

total_stats = []
for activity in df['activity'].unique():
    activity_df = df[df['activity'] == activity]
    
    # Calculate totals
    total_cpu = (activity_df['python_cpu_avg'] * activity_df['duration_seconds']).sum()
    total_memory = (activity_df['python_memory_mb_avg'] * activity_df['duration_seconds']).sum()
    total_disk = activity_df['python_disk_read_mb'].sum() + activity_df['python_disk_write_mb'].sum()
    total_network = activity_df['python_network_sent_mb'].sum() + activity_df['python_network_recv_mb'].sum()
    
    total_stats.append({
        'Activity': activity,
        'Operations': len(activity_df),
        'Total CPU-seconds': total_cpu,
        'Total Memory-MB-seconds': total_memory,
        'Total Disk (MB)': total_disk,
        'Total Network (MB)': total_network,
        'Avg Op Time (s)': activity_df['duration_seconds'].mean(),
    })

total_df = pd.DataFrame(total_stats).sort_values('Total CPU-seconds', ascending=False)
print(total_df.to_string(index=False))

# Per-operation metrics
print("\n\n⚙️ PER-OPERATION METRICS:\n")

per_op_stats = []
for activity in df['activity'].unique():
    activity_df = df[df['activity'] == activity]
    
    per_op_stats.append({
        'Activity': activity,
        'Count': len(activity_df),
        'Avg CPU (%)': activity_df['python_cpu_avg'].mean(),
        'Avg Memory (MB)': activity_df['python_memory_mb_avg'].mean(),
        'Avg Time (s)': activity_df['duration_seconds'].mean(),
        'Avg Disk (MB)': (activity_df['python_disk_read_mb'] + activity_df['python_disk_write_mb']).mean(),
        'Avg Network (MB)': (activity_df['python_network_sent_mb'] + activity_df['python_network_recv_mb']).mean(),
    })

per_op_df = pd.DataFrame(per_op_stats).sort_values('Avg CPU (%)', ascending=False)
print(per_op_df.to_string(index=False))

# Peak resource usage
print("\n\n⚡ PEAK RESOURCE USAGE:\n")

peak_stats = []
for activity in df['activity'].unique():
    activity_df = df[df['activity'] == activity]
    
    peak_stats.append({
        'Activity': activity,
        'Peak CPU (%)': activity_df['python_cpu_max'].max(),
        'Peak Memory (MB)': activity_df['python_memory_mb_max'].max(),
        'Peak GPU (%)': activity_df['python_gpu_max'].max(),
        'Max Duration (s)': activity_df['duration_seconds'].max(),
        'Max Disk Read (MB)': activity_df['python_disk_read_mb'].max(),
        'Max Network (MB)': (activity_df['python_network_sent_mb'] + activity_df['python_network_recv_mb']).max(),
    })

peak_df = pd.DataFrame(peak_stats).sort_values('Peak CPU (%)', ascending=False)
print(peak_df.to_string(index=False))

# Global statistics
print("\n\n🌍 GLOBAL STATISTICS:\n")
print(f"Total operations tracked: {len(df)}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Total analysis period: {(df['timestamp'].max() - df['timestamp'].min()).total_seconds() / 86400:.1f} days")
print(f"\nGlobal peak CPU: {df['python_cpu_max'].max():.1f}%")
print(f"Global peak memory: {df['python_memory_mb_max'].max():.1f} MB")
print(f"Global peak GPU: {df['python_gpu_max'].max():.1f}%")
print(f"Total disk I/O: {(df['python_disk_read_mb'].sum() + df['python_disk_write_mb'].sum()):.1f} MB")
print(f"Total network traffic: {(df['python_network_sent_mb'].sum() + df['python_network_recv_mb'].sum()):.1f} MB")

## 8. Identify Resource Bottlenecks

In [ ]:
def identify_bottlenecks(df: pd.DataFrame) -> Dict[str, List[Dict]]:
    """Identify resource bottlenecks and optimisation opportunities."""
    bottlenecks = {
        'cpu_intensive': [],
        'memory_intensive': [],
        'gpu_intensive': [],
        'disk_intensive': [],
        'network_intensive': [],
        'long_running': [],
    }
    
    # CPU bottlenecks
    high_cpu = df[df['python_cpu_max'] > 50].groupby('activity').size().to_dict()
    for activity, count in high_cpu.items():
        bottlenecks['cpu_intensive'].append({
            'activity': activity,
            'high_cpu_ops': count,
            'pct_of_total': f"{count / len(df[df['activity'] == activity]) * 100:.1f}%"
        })
    
    # Memory bottlenecks
    high_mem = df[df['python_memory_mb_max'] > 500].groupby('activity').size().to_dict()
    for activity, count in high_mem.items():
        bottlenecks['memory_intensive'].append({
            'activity': activity,
            'high_memory_ops': count,
            'pct_of_total': f"{count / len(df[df['activity'] == activity]) * 100:.1f}%"
        })
    
    # GPU bottlenecks
    high_gpu = df[df['python_gpu_max'] > 20].groupby('activity').size().to_dict()
    for activity, count in high_gpu.items():
        bottlenecks['gpu_intensive'].append({
            'activity': activity,
            'high_gpu_ops': count,
            'pct_of_total': f"{count / len(df[df['activity'] == activity]) * 100:.1f}%"
        })
    
    # Disk bottlenecks
    df['total_disk'] = df['python_disk_read_mb'] + df['python_disk_write_mb']
    high_disk = df[df['total_disk'] > 100].groupby('activity').size().to_dict()
    for activity, count in high_disk.items():
        bottlenecks['disk_intensive'].append({
            'activity': activity,
            'high_disk_ops': count,
            'pct_of_total': f"{count / len(df[df['activity'] == activity]) * 100:.1f}%"
        })
    
    # Network bottlenecks
    df['total_network'] = df['python_network_sent_mb'] + df['python_network_recv_mb']
    high_net = df[df['total_network'] > 0.1].groupby('activity').size().to_dict()
    for activity, count in high_net.items():
        bottlenecks['network_intensive'].append({
            'activity': activity,
            'high_network_ops': count,
            'pct_of_total': f"{count / len(df[df['activity'] == activity]) * 100:.1f}%"
        })
    
    # Long-running operations
    long_ops = df[df['duration_seconds'] > df['duration_seconds'].quantile(0.9)].groupby('activity').size().to_dict()
    for activity, count in long_ops.items():
        bottlenecks['long_running'].append({
            'activity': activity,
            'long_ops': count,
            'pct_of_total': f"{count / len(df[df['activity'] == activity]) * 100:.1f}%"
        })
    
    return bottlenecks

bottlenecks = identify_bottlenecks(df)

print("=" * 80)
print("RESOURCE BOTTLENECK ANALYSIS")
print("=" * 80)

if bottlenecks['cpu_intensive']:
    print("\n🔴 CPU BOTTLENECKS:")
    for item in bottlenecks['cpu_intensive']:
        print(f"  {item['activity']}: {item['high_cpu_ops']} high-CPU operations ({item['pct_of_total']})")
else:
    print("\n✅ No significant CPU bottlenecks detected")

if bottlenecks['memory_intensive']:
    print("\n🔴 MEMORY BOTTLENECKS:")
    for item in bottlenecks['memory_intensive']:
        print(f"  {item['activity']}: {item['high_memory_ops']} high-memory operations ({item['pct_of_total']})")
else:
    print("\n✅ No significant memory bottlenecks detected")

if bottlenecks['gpu_intensive']:
    print("\n🟡 GPU BOTTLENECKS:")
    for item in bottlenecks['gpu_intensive']:
        print(f"  {item['activity']}: {item['high_gpu_ops']} GPU-intensive operations ({item['pct_of_total']})")
else:
    print("\n✅ No significant GPU bottlenecks detected")

if bottlenecks['disk_intensive']:
    print("\n🔴 DISK I/O BOTTLENECKS:")
    for item in bottlenecks['disk_intensive']:
        print(f"  {item['activity']}: {item['high_disk_ops']} high-disk operations ({item['pct_of_total']})")
else:
    print("\n✅ No significant disk bottlenecks detected")

if bottlenecks['network_intensive']:
    print("\n🟡 NETWORK BOTTLENECKS:")
    for item in bottlenecks['network_intensive']:
        print(f"  {item['activity']}: {item['high_network_ops']} network-intensive operations ({item['pct_of_total']})")
else:
    print("\n✅ No significant network bottlenecks detected")

if bottlenecks['long_running']:
    print("\n⏱️  LONG-RUNNING OPERATIONS (>90th percentile):")
    for item in bottlenecks['long_running']:
        print(f"  {item['activity']}: {item['long_ops']} long operations ({item['pct_of_total']})")
else:
    print("\n✅ No unusually long operations detected")

print("\n" + "=" * 80)

# Optimisation recommendations
print("\n📋 OPTIMISATION RECOMMENDATIONS:\n")

recommendations = []

# CPU optimisation
if bottlenecks['cpu_intensive']:
    recommendations.append({
        'Priority': 'High',
        'Category': 'CPU',
        'Issue': 'High CPU consumption detected',
        'Recommendation': 'Consider parallelisation, algorithm optimisation, or vectorisation with NumPy',
    })

# Memory optimisation
if bottlenecks['memory_intensive']:
    recommendations.append({
        'Priority': 'High',
        'Category': 'Memory',
        'Issue': 'High memory usage detected',
        'Recommendation': 'Implement streaming/batching, optimise data structures, consider generators',
    })

# Disk optimisation
if bottlenecks['disk_intensive']:
    recommendations.append({
        'Priority': 'Medium',
        'Category': 'Disk I/O',
        'Issue': 'High disk I/O detected',
        'Recommendation': 'Batch small I/O operations, use buffering, consider caching or memory mapping',
    })

# GPU utilisation
cpu_sum = df['python_cpu_avg'].sum()
gpu_sum = df['python_gpu_avg'].sum()
if gpu_sum < cpu_sum * 0.1:
    recommendations.append({
        'Priority': 'Medium',
        'Category': 'GPU',
        'Issue': 'GPU utilisation is low',
        'Recommendation': 'Evaluate opportunities to offload computation to GPU for embedding/inference',
    })

recommendations_df = pd.DataFrame(recommendations)
if not recommendations_df.empty:
    print(recommendations_df.to_string(index=False))
else:
    print("✅ No significant optimisation opportunities identified at this time")

In [ ]:
# Time series analysis of resource trends
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('Resource Utilisation Trends Over Time', fontsize=16, fontweight='bold')

# Filter out invalid timestamps and sort
df_sorted = df[df['timestamp'].notna()].copy()
df_sorted = df_sorted.sort_values('timestamp')
df_sorted['date'] = df_sorted['timestamp'].dt.date

print(f"\n[DATE] Timestamp validation:")
print(f"  Total records: {len(df)}")
print(f"  Valid timestamps: {len(df_sorted)}")
print(f"  Invalid timestamps: {len(df) - len(df_sorted)}")
if len(df_sorted) > 0:
    print(f"  Date range: {df_sorted['date'].min()} to {df_sorted['date'].max()}")

# Daily aggregations
daily_stats = df_sorted.groupby('date').agg({
    'python_cpu_avg': 'mean',
    'python_memory_mb_avg': 'mean',
    'python_gpu_avg': 'mean',
    'duration_seconds': 'mean',
}).reset_index()

if len(daily_stats) == 0:
    print("\n⚠️  No valid data for time series plotting")
    plt.close(fig)
else:
    # Convert date to datetime for proper plotting
    daily_stats['date'] = pd.to_datetime(daily_stats['date'])
    
    # CPU trend
    ax = axes[0, 0]
    ax.plot(daily_stats['date'], daily_stats['python_cpu_avg'], marker='o', linewidth=2, color='#FF6B6B')
    ax.fill_between(daily_stats['date'], daily_stats['python_cpu_avg'], alpha=0.3, color='#FF6B6B')
    ax.set_xlabel('Date')
    ax.set_ylabel('Average CPU (%)')
    ax.set_title('CPU Usage Trend')
    ax.grid(alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Memory trend
    ax = axes[0, 1]
    ax.plot(daily_stats['date'], daily_stats['python_memory_mb_avg'], marker='s', linewidth=2, color='#4ECDC4')
    ax.fill_between(daily_stats['date'], daily_stats['python_memory_mb_avg'], alpha=0.3, color='#4ECDC4')
    ax.set_xlabel('Date')
    ax.set_ylabel('Average Memory (MB)')
    ax.set_title('Memory Usage Trend')
    ax.grid(alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # GPU trend
    ax = axes[1, 0]
    ax.plot(daily_stats['date'], daily_stats['python_gpu_avg'], marker='^', linewidth=2, color='#FFE66D')
    ax.fill_between(daily_stats['date'], daily_stats['python_gpu_avg'], alpha=0.3, color='#FFE66D')
    ax.set_xlabel('Date')
    ax.set_ylabel('Average GPU (%)')
    ax.set_title('GPU Usage Trend')
    ax.grid(alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Duration trend
    ax = axes[1, 1]
    ax.plot(daily_stats['date'], daily_stats['duration_seconds'], marker='D', linewidth=2, color='#C7CEEA')
    ax.fill_between(daily_stats['date'], daily_stats['duration_seconds'], alpha=0.3, color='#C7CEEA')
    ax.set_xlabel('Date')
    ax.set_ylabel('Average Duration (seconds)')
    ax.set_title('Operation Duration Trend')
    ax.grid(alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Time series visualisations created")

## Summary And Export

Complete resource utilisation analysis has been generated. The notebook provides:

- **Activity-level breakdowns** of CPU, memory, GPU, disk, and network consumption
- **Comparative analysis** showing resource usage patterns across different operations
- **Visualisations** including bar charts, box plots, and time series trends
- **Bottleneck identification** for activities with high resource consumption
- **Optimisation recommendations** based on detected patterns
- **Global statistics** and efficiency metrics for system-wide analysis

Use this analysis to identify optimisation opportunities and resource allocation strategies.

In [ ]:
# Export analysis results to CSV
export_dir = Path('resource_analysis_export')
export_dir.mkdir(exist_ok=True)

# Export summary tables
activity_summary.to_csv(export_dir / 'activity_summary.csv', index=False)
total_df.to_csv(export_dir / 'total_resource_consumption.csv', index=False)
per_op_df.to_csv(export_dir / 'per_operation_metrics.csv', index=False)
peak_df.to_csv(export_dir / 'peak_resource_usage.csv', index=False)
efficiency_df.to_csv(export_dir / 'efficiency_metrics.csv', index=False)

# Export raw data
df.to_csv(export_dir / 'full_dataset.csv', index=False)

print(f"✓ Exported analysis results to {export_dir}/")
print(f"\nExported files:")
print(f"  - activity_summary.csv")
print(f"  - total_resource_consumption.csv")
print(f"  - per_operation_metrics.csv")
print(f"  - peak_resource_usage.csv")
print(f"  - efficiency_metrics.csv")
print(f"  - full_dataset.csv")